# pancreatic TLS-associated spatial phenotype network

This tutorial is designed for biological research teams and clinical teams, demonstrating how to use the tools in this repository to:：
**Lesion segmentation**：CT image classification and segmentation based on nnUNetv2
**Subtype classification**：Based on CT-CLIP, the four classifications (DE/DIS/EX/ST)

## 教程结构

| part | content | suitable for the croud |
| :--- | :--- | :--- |
| **Part A** | Fast reasoning (using pre-trained weights) | Clinical team, quickly get familiar with and use the pre-trained weights |
| **Part B** | Model training (using sample data) | Research team, customized training |
| **Part C** | Public dataset guidelines (OBIA0007), custom training | Large-scale data training |

> 💡 **Strong recommendation**：First, complete Part A to ensure the environment is correct, and then attempt Part B and Part C.

In [ ]:
import os
import sys
import torch
import subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
import json
import pandas as pd

print("=" * 60)
print("0. environmental testing")
print("=" * 60)

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()} (To be used {'GPU' if torch.cuda.is_available() else 'CPU'})")

# check nnUNet v2
try:
    import nnunetv2
    print("✅ nnUNet v2 have already installed")
except ImportError:
    print("❌ nnUNet v2 Not installed. Please proceed.: pip install nnunetv2")

# check medpy
try:
    import medpy
    print("✅ MedPy have already installed")
except ImportError:
    print("❌ MedPy Not installed. Please proceed: pip install medpy")

# check CT-CLIP 
if Path("CT-CLIP").exists():
    print("✅ CT-CLIP The source code is ready")
else:
    print("⚠️ not found CT-CLIP，The classification section requires : git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git")

#Check item script
scripts = ["segmentation/infer.py", "process.py", "infer.py", "main.py"]
for s in scripts:
    if Path(s).exists():
        print(f"✅ {s} is ok")
    else:
        print(f"⚠️ not found {s}")

## 1. Data Preparation

This tutorial uses 3 example CT scans (approximately 200 MB in total), including：
- `example_data/images/`：3 CT images（.nii.gz）
- `example_data/labels/`：Corresponding segmentation labels（.nii.gz）
- `labels.xlsx`：Sample labels（for classification）

> If you have not yet downloaded the example data and pretrained model weights, please refer to the download links in the README.

### Data Format Description

| Data Type | Format | Naming Convention | Description |
| :--- | :--- | :--- | :--- |
|CT Image | .nii.gz | `{case_id}.nii.gz` or `{case_id}_0000.nii.gz` | 3D CT scan |
|Segmentation Label | .nii.gz | `{case_id}.nii.gz` | 0=background，1=lesion |
|Classification Label | .xlsx | `labels.xlsx` |Contains sample_id 和 type columns|

In [ ]:
print("\n" + "=" * 60)
print("1. database integrity checking")
print("=" * 60)

# Check the sample data
IMAGE_DIR = "example_data/images"
LABEL_DIR = "example_data/labels"
LABEL_XLSX = "labels.xlsx"

if Path(IMAGE_DIR).exists():
    images = list(Path(IMAGE_DIR).glob("*.nii.gz"))
    print(f"✅ find {len(images)}  CT image")
    for f in images:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"   - {f.name} ({size_mb:.1f} MB)")
else:
    print(f"❌ No image directory found: {IMAGE_DIR}")

if Path(LABEL_DIR).exists():
    labels = list(Path(LABEL_DIR).glob("*.nii.gz"))
    print(f"✅ find {len(labels)} label")
else:
    print(f"⚠️ No tag directory found: {LABEL_DIR}（The reasoning can still be carried out, but it is impossible to make an assessment.）")

if Path(LABEL_XLSX).exists():
    df = pd.read_excel(LABEL_XLSX)
    print(f"✅ Find the classification label: {LABEL_XLSX} ({len(df)} item)")
    print(f"   column name: {list(df.columns)}")
else:
    print(f"⚠️ No classification label was found: {LABEL_XLSX}")

# Part A: Quick Inference

This section uses the provided pretrained weights to directly perform segmentation and classification inference on the example data.

In [ ]:
print("\n" + "=" * 60)
print("Part A: "Quick Inference")
print("=" * 60)

# ===== seg configuration =====
SEG_WEIGHTS = "models/segmentation_model.pth"
SEG_IMAGE_DIR = "example_data/images"
SEG_LABEL_DIR = "example_data/labels"
SEG_OUTPUT_DIR = "results_seg"

# ===== cls configuration =====
CLS_LABEL_XLSX = "labels.xlsx"
CLS_IMAGE_DIR = "example_data/images"
CLS_LABEL_DIR = "example_data/labels"
CLS_PRETRAINED = "models/ct_clip/CT_VocabFine_v2.pt"
CLS_CKPT = "models/classifier/best_classifier.pth"
CLS_FEATURE_DIR = "features"
CLS_OUTPUT_DIR = "results_cls"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


os.makedirs(SEG_OUTPUT_DIR, exist_ok=True)
os.makedirs(CLS_OUTPUT_DIR, exist_ok=True)
os.makedirs(CLS_FEATURE_DIR, exist_ok=True)

print(f"✅ DEVICE: {DEVICE}")
print(f"✅ seg out: {SEG_OUTPUT_DIR}")
print(f"✅ cls out: {CLS_OUTPUT_DIR}")

## A.1 Lesion Segmentation Inference

Run `segmentation/infer.py`to perform segmentation on the 3 example CT scans. Dice/HD95 metrics will be automatically computed if labels are available.

In [ ]:
print("\n A.1 Running segmentation inference (approximately 2-3 minutes)...")

if not Path(SEG_WEIGHTS).exists():
    print(f"Segmentation weights not found: {SEG_WEIGHTS}")
    print(" Please download the weights file first")
else:
    seg_cmd = [
        "python", "segmentation/infer.py",
        "--weights", SEG_WEIGHTS,
        "--image_dir", SEG_IMAGE_DIR,
        "--output_dir", SEG_OUTPUT_DIR,
        "--device", DEVICE
    ]
    
    if Path(SEG_LABEL_DIR).exists() and list(Path(SEG_LABEL_DIR).glob("*.nii.gz")):
        seg_cmd.extend(["--label_dir", SEG_LABEL_DIR])
        print("✅ When a label is detected, the evaluation indicators will be automatically calculated.")
    
    print("执行命令:", " ".join(seg_cmd))
    print("-" * 50)
    
    result = subprocess.run(seg_cmd, capture_output=False, text=True)
    print(f"\n✅ Segmentation completed! The results are saved in: {SEG_OUTPUT_DIR}/")

In [ ]:
metrics_file = Path(SEG_OUTPUT_DIR) / "metrics_summary.json"

if metrics_file.exists():
    with open(metrics_file, "r") as f:
        data = json.load(f)
    
    summary = data.get("summary", {})
    per_case = data.get("per_case", [])
    
    print(f"  Dice:  {summary.get('mean_dice', 0):.4f} ± {summary.get('std_dice', 0):.4f}")
    print(f"  HD95:  {summary.get('mean_hd95', 0):.2f} ± {summary.get('std_hd95', 0):.2f}")
    for item in per_case:
        print(f"  {item['case']}: Dice={item['dice']:.4f}, HD95={item['hd95']:.2f}")
else:
    print("No evaluation indicators were found")

## A.2 Subtype Classification Inference

1. Use CT-CLIP to extract visual features
2. Use the pretrained weights for four-class classification 

In [ ]:
print("\n A.2.1 Extracting CT-CLIP features（approximately 5-10 minutes）...")

if not Path("CT-CLIP").exists():
    print("  CT-CLIP source code not found，Please run: git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git")
    skip_cls = True
elif not Path(CLS_PRETRAINED).exists():
    print(f" CT-CLIP weights not found: {CLS_PRETRAINED}")
    skip_cls = True
else:
    skip_cls = False
    sys.path.append(os.path.abspath("."))
    
    process_cmd = [
        "python", "process.py",
        "--input_dir", CLS_IMAGE_DIR,
        "--label_dir", CLS_LABEL_DIR,
        "--output_dir", CLS_FEATURE_DIR,
        "--pretrained_path", CLS_PRETRAINED,
        "--device", DEVICE
    ]
    
    print("run:", " ".join(process_cmd))
    print("-" * 50)
    
    if Path("process.py").exists():
        subprocess.run(process_cmd, capture_output=False, text=True)
    else:
        print("not found process.py")
        skip_cls = True

In [ ]:
if not skip_cls:
    print("\n A.2.2 Classification reasoning is not running....")
    
    if not Path(CLS_CKPT).exists():
        print(f" Classifier weights not found: {CLS_CKPT}")
        print("  Please complete Part B training first, or download the pretrained weights")
    else:
        CLS_OUTPUT = f"{CLS_OUTPUT_DIR}/predictions.txt"
        
        infer_cmd = [
            "python", "infer.py",
            "--ckpt", CLS_CKPT,
            "--label_xlsx", CLS_LABEL_XLSX,
            "--feature_dir", CLS_FEATURE_DIR,
            "--output", CLS_OUTPUT,
            "--device", DEVICE
        ]
        
        print("Command not executed:", " ".join(infer_cmd))
        print("-" * 50)
        
        if Path("infer.py").exists():
            subprocess.run(infer_cmd, capture_output=False, text=True)
            print(f" Classification reasoning completed！The result is saved in: {CLS_OUTPUT}")
        else:
            print(" not found infer.py")

In [ ]:
if not skip_cls:
    CLS_OUTPUT = f"{CLS_OUTPUT_DIR}/predictions.txt"
    
    if Path(CLS_OUTPUT).exists():
        df = pd.read_csv(CLS_OUTPUT, sep='\t')
        print("\n" + "=" * 60)
        print("A.2 Results")
        print("=" * 60)
        print(f"Total samples: {len(df)}")
        print("\n top 5 rows：")
        print(df.head())
        
        print("\n Prediction class distribution：")
        print(df['pred_label'].value_counts())
        
        if 'gt_label' in df.columns:
            acc = (df['gt_label'] == df['pred_label']).mean()
            print(f"\n Acc: {acc:.4f}")
    else:
        print(" Classification results not found")

# Part B: Model Training

This section demonstrates how to retrain the models using the example data.

**For**：Research teams and users who wish to train models with custom data.

## B.1 Segmentation Model Training（nnUNet v2）
nnUNet v2 training uses the official commands. You need to:

1. Convert data to nnUNet v2 format
2. Run the training command
3. Export the best model

> 💡 The example data in this tutorial contains only 3 cases, which is suitable for quickly validating the workflow. For production use, we recommend using more data (>500 cases) to achieve good performance.

In [ ]:
print("\n" + "=" * 60)
print("=" * 60)

print(" nnUNet v2 The training of nnUNet v2 requires the following environment variables to be set：")
print("  export nnUNet_raw='/path/to/nnUNet_raw'")
print("  export nnUNet_preprocessed='/path/to/nnUNet_preprocessed'")
print("  export nnUNet_results='/path/to/nnUNet_results'")

print("\n Dataset format requirements：")
print("  nnUNet_raw/DatasetXXX_YYY/")
print("  ├── imagesTr/")
print("  │   ├── case_001_0000.nii.gz")
print("  │   └── case_002_0000.nii.gz")
print("  └── labelsTr/")
print("      ├── case_001.nii.gz")
print("      └── case_002.nii.gz")

print("  python utils/convert_to_nnunet.py \\")
print("      --input_dir example_data/images \\")
print("      --label_dir example_data/labels \\")
print("      --output_dir /path/to/nnUNet_raw/DatasetXXX_YYY")

In [ ]:
print("\n" + "=" * 60)
print("B.1 nnUNet v2 Training Commands")
print("=" * 60)

print(""" 1. Dataset planning（create Dataset folder）：
nnUNetv2_plan_and_preprocess -d XXX --verify_dataset_integrity

 2. Start training (3D full resolution):：
nnUNetv2_train XXX 3d_fullres 0 --npz

 3. Cross-validation (optional)：
nnUNetv2_train XXX 3d_fullres 0 -val

 4. Find the best model：
nnUNetv2_find_best_configuration XXX -c 3d_fullres

 5. Export model weights：
nnUNetv2_export_model_to_zip \\
    -d XXX \\
    -c 3d_fullres \\
    -o /path/to/exported_model.zip
""")

print("Detailed documentation：https://github.com/MIC-DKFZ/nnUNet")

In [ ]:
## B.2 Classification Model Training

Use the provided `main.py` script to train the MLP classification head.

**Complete workflow:**
1. Extract CT-CLIP features (`process.py`)
2. Train the MLP classifier (`main.py`)
3. Run inference and evaluate the model (`infer.py`)

In [ ]:
print("\n" + "=" * 60)
print("B.2.1 Extracting CT-CLIP Features")
print("=" * 60)

if not Path(CLS_PRETRAINED).exists():
    print(f" CT-CLIP weights not found: {CLS_PRETRAINED}")
    print("  Please download the weights file first")
else:
    print("\n Extracting features (approximately 5-10 minutes)...")
    
    process_cmd = [
        "python", "process.py",
        "--input_dir", CLS_IMAGE_DIR,
        "--label_dir", CLS_LABEL_DIR,
        "--output_dir", CLS_FEATURE_DIR,
        "--pretrained_path", CLS_PRETRAINED,
        "--device", DEVICE
    ]
    
    print("executive command:", " ".join(process_cmd))
    print("-" * 50)
    
    if Path("process.py").exists():
        subprocess.run(process_cmd, capture_output=False, text=True)
        print(" Command for completing feature extraction！")
    else:
        print(" not found process.py")

In [ ]:
print("\n" + "=" * 60)
print("B.2.2 Training the MLP Classifier")
print("=" * 60)

# Check feature files
feature_files = list(Path(CLS_FEATURE_DIR).glob("*.npy"))
if len(feature_files) < 3:
    print(f" Feature files insufficient（currently {len(feature_files)} ，Please complete feature extraction first")
else:
    print(f"  {len(feature_files)} feature files")
    
    # Training configuration
    TRAIN_SAVE_DIR = "models/classifier_trained"
    os.makedirs(TRAIN_SAVE_DIR, exist_ok=True)
    
    print("\n Start the training (which will take approximately 5 to 10 minutes)...")
    
    train_cmd = [
        "python", "main.py",
        "--label_xlsx", CLS_LABEL_XLSX,
        "--feature_dir", CLS_FEATURE_DIR,
        "--save_dir", TRAIN_SAVE_DIR,
        "--device", DEVICE,
        "--epochs", "30",
        "--batch_size", "8",
        "--early_stopping_patience", "10"
    ]
    
    print("executive command:", " ".join(train_cmd))
    print("-" * 50)
    
    if Path("main.py").exists():
        subprocess.run(train_cmd, capture_output=False, text=True)
        print(f"\n Training completed! The model has been saved as: {TRAIN_SAVE_DIR}/")
        print("   best: best_classifier.pth")
    else:
        print(" not found main.py")

In [ ]:
print("\n" + "=" * 60)
print("B.2.3 Evaluate the trained model")
print("=" * 60)

TRAINED_CKPT = f"{TRAIN_SAVE_DIR}/best_classifier.pth"

if Path(TRAINED_CKPT).exists():
    print("\n Perform inference using the trained model...")
    
    CLS_OUTPUT_TRAINED = f"{CLS_OUTPUT_DIR}/predictions_trained.txt"
    
    infer_cmd = [
        "python", "infer.py",
        "--ckpt", TRAINED_CKPT,
        "--label_xlsx", CLS_LABEL_XLSX,
        "--feature_dir", CLS_FEATURE_DIR,
        "--output", CLS_OUTPUT_TRAINED,
        "--device", DEVICE
    ]
    
    print("executive command:", " ".join(infer_cmd))
    print("-" * 50)
    
    if Path("infer.py").exists():
        subprocess.run(infer_cmd, capture_output=False, text=True)
        
        if Path(CLS_OUTPUT_TRAINED).exists():
            df = pd.read_csv(CLS_OUTPUT_TRAINED, sep='\t')
            print("\n  Classification results of the model after training：")
            print(f"Total sample size: {len(df)}")
            print("\n Predicting the distribution of categories：")
            print(df['pred_label'].value_counts())
            
            if 'gt_label' in df.columns:
                acc = (df['gt_label'] == df['pred_label']).mean()
                print(f"\n  acc: {acc:.4f}")
else:
    print(f" The trained model was not found: {TRAINED_CKPT}")

# Part C: Public Dataset Guide (OBIA0007)

The example data in this tutorial contains only 3 cases, which is sufficient for quickly demonstrating the workflow.

For full-scale training and validation, we recommend using the public dataset **OBIA0007**.

## C.1 Dataset Overview

| Attribute | Details |
| :--- | :--- |
| **Dataset Name** | OBIA0007 |
| **Disease** | Pancreatic Cancer |
| **Modality** | CT |
| **Sample Size** | 500 individuals, 149,001 images |

## C.2 Data Access

Visit: https://ngdc.cncb.ac.cn/obia/collection/OBIA0007

1. Register for an NGDC account
2. Agree to the data usage agreement
3. Download the data

## C.4 Full-Scale Training

### Segmentation Training (nnUNet v2)

# 1. Prepare data (nnUNet format)
# 2. Preprocess
nnUNetv2_plan_and_preprocess -d DATASET_ID

# 3. Train (recommended 5-fold cross-validation)
for fold in 0 1 2 3 4; do
    nnUNetv2_train DATASET_ID 3d_fullres $fold --npz
done

# 4. Find the best configuration
nnUNetv2_find_best_configuration DATASET_ID -c 3d_fullres

## C.4 Full-Scale Training

### Classification Training

# 1. Extract features
python process.py \
    --input_dir /path/to/500_ct \
    --label_dir /path/to/500_labels \
    --output_dir features_500

# 2. Train the classifier
python main.py \
    --label_xlsx labels_500.xlsx \
    --feature_dir features_500 \
    --epochs 200 \
    --batch_size 64

# 3. Run inference
python infer.py \
    --ckpt models/classifier_trained/best_classifier.pth \
    --label_xlsx labels_500.xlsx \
    --feature_dir features_500

## C.5 Citation Information

If you use the OBIA0007 dataset in your work, please cite:

> [OBIA0007] Fudan University Shanghai Cancer Center Pancreatic Cancer CT Imaging Dataset
> https://ngdc.cncb.ac.cn/obia/collection/OBIA0007

If you use this tool in your work, please cite:

- **nnUNet v2**: https://github.com/MIC-DKFZ/nnUNet
- **CT-CLIP**: https://arxiv.org/abs/2403.17834
- **OBIA0007**: https://ngdc.cncb.ac.cn/obia/collection/OBIA0007

If you have any questions, please refer to README.md or contact the project maintainer.